# Blood cell type classification

### Imports

In [4]:
import os
import torch
import cv2
import glob
import matplotlib as plt
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import kagglehub

Utils functions

In [5]:
def calculate_mean_and_std(dataloader):
    channels_sum = torch.zeros(3)
    channels_squared_sum = torch.zeros(3)
    total_pixels = 0

    for images, _ in dataloader:
        batch_pixels = images.size(0) * images.size(2) * images.size(3)
        total_pixels += batch_pixels

        channels_sum += torch.sum(images, dim=[0, 2, 3])
        channels_squared_sum += torch.sum(images ** 2, dim=[0, 2, 3])

    mean = channels_sum / total_pixels
    std = torch.sqrt((channels_squared_sum / total_pixels) - (mean ** 2))

    return mean, std

### Loading in the dataset

In [6]:
path = kagglehub.dataset_download("paultimothymooney/blood-cells")

100%|██████████| 108M/108M [00:01<00:00, 107MB/s] 

Extracting files...


In [7]:
train_dir = f"{path}/dataset2-master/dataset2-master/images/TRAIN"
test_dir = f"{path}/dataset2-master/dataset2-master/images/TEST"

data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Since dataset has the PyTorch-friendly structure (each class has its
# own subfolder within TRAIN and TEST folders) we can just use the
# .ImageFolder() method to create PyTorch Datasets instances.
train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Calculating mean and standard deviation of the dataset images
mean, std = calculate_mean_and_std(train_loader)

In [8]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean.tolist(), std=std.tolist())
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean.tolist(), std=std.tolist())
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)